# 03 · Correlation Analysis

Compute pairwise correlations between extracted LENR parameters across classified corpus documents.

> **[MOCK]** Field names and thresholds are placeholders pending the physicist ontology session.  
> Update `PARAM_COLS` after real field names are defined in the domain pack.

In [ ]:
import os, json, requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from dotenv import load_dotenv

load_dotenv()
API_BASE = os.getenv('VERITAS_API_BASE', 'http://localhost:5000')
CORPUS_ID = os.getenv('VERITAS_CORPUS_ID', 'demo-lenr-corpus')  # [MOCK]

In [ ]:
# [MOCK] Parameter columns — update to match real LENR ontology field names after physicist session
PARAM_COLS = ['excess_heat_watts', 'cop', 'loading_ratio', 'current_density_ma_cm2']

resp = requests.get(f'{API_BASE}/api/corpora/{CORPUS_ID}/documents')
if resp.ok:
    documents = resp.json()
else:
    # [MOCK] Fallback synthetic data — remove when real extraction pipeline is connected
    rng = np.random.default_rng(42)
    n = 30
    loading = rng.uniform(0.7, 0.95, n)
    cop = 1.0 + 0.8 * (loading - 0.7) / 0.25 + rng.normal(0, 0.1, n)
    heat = cop * rng.uniform(3, 10, n)
    current = rng.uniform(50, 300, n)
    documents = [
        {'document_id': f'doc-{i:03d}', 'extracted_fields': {
            'excess_heat_watts': float(heat[i]),
            'cop': float(cop[i]),
            'loading_ratio': float(loading[i]),
            'current_density_ma_cm2': float(current[i])
        }} for i in range(n)
    ]
    print('[MOCK] Using synthetic data')

rows = []
for doc in documents:
    row = {'document_id': doc['document_id']}
    row.update(doc.get('extracted_fields', {}))
    rows.append(row)

df = pd.DataFrame(rows)
available = [c for c in PARAM_COLS if c in df.columns]
print(f'{len(df)} documents, {len(available)} parameter columns')

In [ ]:
# Pairwise Pearson correlation matrix
if len(available) >= 2:
    corr = df[available].corr(method='pearson')
    
    fig, ax = plt.subplots(figsize=(7, 6))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                vmin=-1, vmax=1, center=0, ax=ax, square=True)
    ax.set_title('Parameter Correlation Matrix (Pearson)')
    plt.tight_layout()
    plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1))

In [ ]:
# Scatter plots for strongest correlations
if len(available) >= 2:
    pairs = [(a, b) for i, a in enumerate(available) for b in available[i+1:]]
    strong = sorted(pairs, key=lambda p: abs(corr.loc[p[0], p[1]]), reverse=True)[:3]
    
    fig, axes = plt.subplots(1, len(strong), figsize=(5 * len(strong), 4))
    if len(strong) == 1: axes = [axes]
    
    for ax, (x, y) in zip(axes, strong):
        slope, intercept, r, p, _ = stats.linregress(df[x].dropna(), df[y].dropna())
        ax.scatter(df[x], df[y], alpha=0.6, s=30)
        xs = np.linspace(df[x].min(), df[x].max(), 100)
        ax.plot(xs, slope * xs + intercept, 'r-', linewidth=1.5)
        ax.set_xlabel(x); ax.set_ylabel(y)
        ax.set_title(f'r={r:.2f}, p={p:.3f}')
    
    plt.suptitle('Strongest Pairwise Correlations', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('top_correlations.png', dpi=150, bbox_inches='tight')
    plt.show()